In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning) 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns
sns.set_style("whitegrid")

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier

In [3]:
import pandas as pd
import numpy as np

def introduce_missing(df, column, missing_rate, random_state=42):
    """
    Добавляет пропуски (NaN) в указанный столбец датафрейма.

    Parameters:
    -----------
    df : pd.DataFrame
        Исходный датафрейм.
    column : str
        Имя столбца, в который нужно добавить пропуски.
    missing_rate : float
        Доля пропусков (от 0.0 до 1.0).
    random_state : int
        Для воспроизводимости.

    Returns:
    --------
    df_missing : pd.DataFrame
        Датафрейм с добавленными пропусками (оригинал не изменяется).
    """
    # Проверки
    if column not in df.columns:
        raise ValueError(f"Столбец '{column}' не найден в датафрейме. Доступные столбцы: {list(df.columns)}")

    if not (0.0 <= missing_rate <= 1.0):
        raise ValueError(f"missing_rate должен быть от 0 до 1. Получено: {missing_rate}")

    if missing_rate == 1.0 and len(df) > 0:
        print("⚠️ Предупреждение: ты делаешь пропуски во всех строках столбца. Модели это не оценят.")

    if missing_rate == 0.0:
        print("✓ Пропуски не добавлены (missing_rate = 0). Возвращаю копию исходного датафрейма.")
        return df.copy()

    # Копируем, чтобы не мутировать оригинал
    df_missing = df.copy()

    n_rows = len(df_missing)
    n_missing = int(np.floor(n_rows * missing_rate))

    # Выбираем случайные индексы
    rng = np.random.default_rng(random_state)
    missing_indices = rng.choice(df_missing.index, size=n_missing, replace=False)

    # Вставляем NaN
    df_missing.loc[missing_indices, column] = np.nan

    # Статистика для отчёта
    actual_rate = n_missing / n_rows
    print(f"✓ Добавлено {n_missing} пропусков ({actual_rate:.1%}) в столбец '{column}'.")

    return df_missing


In [23]:
import pandas as pd
import numpy as np

# Фиксируем seed для воспроизводимости
np.random.seed(42)

n = 2000

# Генерируем предикторы
machine_id = np.random.randint(0, 100, n)
hour = np.random.randint(6, 24, n)
day_type = np.random.choice([0, 1], n, p=[0.7, 0.3])
drinks_sold = np.random.poisson(12, n).clip(0, 40)
beans_level = np.random.uniform(10, 100, n).round(1)
milk_level = np.random.uniform(0, 100, n).round(1)
pressure = np.random.uniform(2.0, 9.0, n).round(2)
last_clean_hours = np.random.randint(1, 121, n)

# Генерируем water_temp (связь с предикторами + шум)
# Базовая температура: в часы пик выше, после очистки – выше, от давления слабо
base_temp = (
    70
    + 0.5 * drinks_sold
    - 0.2 * (24 - hour) ** 0.7
    + 2.0 * day_type
    + 1.5 * (pressure - 5.0)
    + 0.1 * (120 - last_clean_hours)
    + np.random.normal(0, 3.0, n)
)
water_temp = base_temp.clip(60.0, 98.0).round(1)

# Генерируем таргеты (зависят от предикторов, в первую очередь — от температуры)
# brew_time: чем горячее вода — тем быстрее заваривается, но нелинейно
brew_time_base = (
    40
    - 0.35 * water_temp
    + 0.02 * water_temp ** 0.5 * drinks_sold
    - 0.5 * pressure
    + 2.0 * (120 - last_clean_hours) / 120
    + np.random.normal(0, 2.5, n)
)
brew_time = brew_time_base.clip(15, 55).round(1)

# water_consumption: горячая вода немного меньше расходуется (быстрее пар),
# плюс влияют давление, количество порций (больше — выше разовый расход из-за паровых потерь)
water_cons_base = (
    0.30
    - 0.002 * water_temp
    + 0.025 * (drinks_sold / 12)
    + 0.0075 * pressure
    + 0.03 * day_type
    + np.random.normal(0, 0.03, n)
)
water_consumption = water_cons_base.clip(0.15, 0.45).round(3)

# Формируем итоговый DataFrame
df = pd.DataFrame({
    'machine_id': machine_id,
    'hour': hour,
    'day_type': day_type,
    'drinks_sold': drinks_sold,
    'beans_level': beans_level,
    'milk_level': milk_level,
    'pressure': pressure,
    'last_clean_hours': last_clean_hours,
    'water_temp': water_temp,
    'brew_time': brew_time,
    'water_consumption': water_consumption,
})

# Смотрим на результат
print(df.head(10))
print(f"\nФорма: {df.shape}")
print(f"Пропусков в water_temp: {df['water_temp'].isna().sum()}")
print(f"Диапазон brew_time: {df['brew_time'].min():.1f} – {df['brew_time'].max():.1f}")
print(f"Диапазон water_consumption: {df['water_consumption'].min():.3f} – {df['water_consumption'].max():.3f}")


   machine_id  hour  day_type  drinks_sold  beans_level  milk_level  pressure  \
0          51    17         0           15         80.4        70.2      3.58   
1          92    19         1            6         59.0        49.6      5.74   
2          14    20         0           11         23.9        43.1      6.87   
3          71    21         0           14         32.6        95.4      7.42   
4          60     6         0           16         76.5        20.6      2.30   
5          20    15         0           16         82.5        31.2      5.00   
6          82    13         0            5         35.1        30.8      2.86   
7          86    23         0           14         43.1        28.9      5.02   
8          74     6         0            8         17.0        46.9      5.04   
9          74     7         1           11         74.2        98.9      5.49   

   last_clean_hours  water_temp  brew_time  water_consumption  
0                 4        84.1       15.0  

In [24]:
df

,machine_id,hour,day_type,drinks_sold,beans_level,milk_level,pressure,last_clean_hours,water_temp,brew_time,water_consumption
0,51,17,0,15,80.4,70.2,3.58,4,84.1,15.0,0.219
1,92,19,1,6,59.0,49.6,5.74,106,72.9,15.0,0.254
2,14,20,0,11,23.9,43.1,6.87,33,87.8,15.0,0.182
3,71,21,0,14,32.6,95.4,7.42,43,91.0,15.0,0.193
4,60,6,0,16,76.5,20.6,2.30,112,70.8,18.1,0.166
...,...,...,...,...,...,...,...,...,...,...,...
1995,85,7,1,11,46.4,4.0,7.72,117,80.4,15.0,0.238
1996,50,15,0,14,66.2,23.6,7.15,97,79.9,15.0,0.226
1997,87,11,0,20,90.0,37.4,7.39,94,85.9,15.0,0.259
1998,40,16,1,17,28.9,18.3,3.62,21,87.5,15.0,0.194


In [25]:
df = introduce_missing(df, column='water_temp', missing_rate=0.12, random_state=42)
df

✓ Добавлено 240 пропусков (12.0%) в столбец 'water_temp'.


,machine_id,hour,day_type,drinks_sold,beans_level,milk_level,pressure,last_clean_hours,water_temp,brew_time,water_consumption
0,51,17,0,15,80.4,70.2,3.58,4,84.1,15.0,0.219
1,92,19,1,6,59.0,49.6,5.74,106,72.9,15.0,0.254
2,14,20,0,11,23.9,43.1,6.87,33,87.8,15.0,0.182
3,71,21,0,14,32.6,95.4,7.42,43,91.0,15.0,0.193
4,60,6,0,16,76.5,20.6,2.30,112,70.8,18.1,0.166
...,...,...,...,...,...,...,...,...,...,...,...
1995,85,7,1,11,46.4,4.0,7.72,117,80.4,15.0,0.238
1996,50,15,0,14,66.2,23.6,7.15,97,79.9,15.0,0.226
1997,87,11,0,20,90.0,37.4,7.39,94,85.9,15.0,0.259
1998,40,16,1,17,28.9,18.3,3.62,21,87.5,15.0,0.194


In [17]:
# df.to_csv('data/hotel_missing_stars.csv', index = False)

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   machine_id         2000 non-null   int32  
 1   hour               2000 non-null   int32  
 2   day_type           2000 non-null   int32  
 3   drinks_sold        2000 non-null   int32  
 4   beans_level        2000 non-null   float64
 5   milk_level         2000 non-null   float64
 6   pressure           2000 non-null   float64
 7   last_clean_hours   2000 non-null   int32  
 8   water_temp         1760 non-null   float64
 9   brew_time          2000 non-null   float64
 10  water_consumption  2000 non-null   float64
dtypes: float64(6), int32(5)
memory usage: 132.9 KB


In [27]:
a = df.iloc[range(1800)]
a

,machine_id,hour,day_type,drinks_sold,beans_level,milk_level,pressure,last_clean_hours,water_temp,brew_time,water_consumption
0,51,17,0,15,80.4,70.2,3.58,4,84.1,15.0,0.219
1,92,19,1,6,59.0,49.6,5.74,106,72.9,15.0,0.254
2,14,20,0,11,23.9,43.1,6.87,33,87.8,15.0,0.182
3,71,21,0,14,32.6,95.4,7.42,43,91.0,15.0,0.193
4,60,6,0,16,76.5,20.6,2.30,112,70.8,18.1,0.166
...,...,...,...,...,...,...,...,...,...,...,...
1795,26,17,0,15,67.1,80.9,7.72,3,94.9,15.0,0.204
1796,37,18,1,11,79.5,55.2,8.08,41,91.7,15.0,0.241
1797,59,12,0,16,39.2,31.7,5.79,8,93.3,15.0,0.189
1798,6,20,0,10,76.6,12.6,4.29,90,76.4,18.1,0.213


In [33]:
# a.to_csv('data/vending.csv', index = False)

In [34]:
b = df.iloc[range(1800, len(df))]
b.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 200 entries, 1800 to 1999
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   machine_id         200 non-null    int32  
 1   hour               200 non-null    int32  
 2   day_type           200 non-null    int32  
 3   drinks_sold        200 non-null    int32  
 4   beans_level        200 non-null    float64
 5   milk_level         200 non-null    float64
 6   pressure           200 non-null    float64
 7   last_clean_hours   200 non-null    int32  
 8   water_temp         179 non-null    float64
 9   brew_time          200 non-null    float64
 10  water_consumption  200 non-null    float64
dtypes: float64(6), int32(5)
memory usage: 14.8 KB


In [35]:
b = b.drop(columns = ['brew_time', 'water_consumption'])
b

,machine_id,hour,day_type,drinks_sold,beans_level,milk_level,pressure,last_clean_hours,water_temp
1800,44,13,0,10,41.1,52.5,5.07,75,72.9
1801,57,10,1,9,40.6,70.2,4.01,114,71.3
1802,30,19,1,6,91.0,97.0,3.07,63,73.1
1803,22,9,0,19,14.5,43.5,5.70,65,82.2
1804,41,15,1,14,41.5,40.6,6.16,79,80.5
...,...,...,...,...,...,...,...,...,...
1995,85,7,1,11,46.4,4.0,7.72,117,80.4
1996,50,15,0,14,66.2,23.6,7.15,97,79.9
1997,87,11,0,20,90.0,37.4,7.39,94,85.9
1998,40,16,1,17,28.9,18.3,3.62,21,87.5


In [36]:
# b.to_csv('data/vending_new.csv', index=False)

In [22]:
d = df.dropna()
# d['stars'] = d['stars'].astype(str)
d

,machine_id,hour,day_type,drinks_sold,beans_level,milk_level,pressure,last_clean_hours,water_temp,brew_time,water_consumption
0,51,17,0,15,80.4,70.2,3.58,4,84.1,15.0,0.219
1,92,19,1,6,59.0,49.6,5.74,106,72.9,15.0,0.254
2,14,20,0,11,23.9,43.1,6.87,33,87.8,15.0,0.182
3,71,21,0,14,32.6,95.4,7.42,43,91.0,15.0,0.193
4,60,6,0,16,76.5,20.6,2.30,112,70.8,18.1,0.166
...,...,...,...,...,...,...,...,...,...,...,...
1995,85,7,1,11,46.4,4.0,7.72,117,80.4,15.0,0.238
1996,50,15,0,14,66.2,23.6,7.15,97,79.9,15.0,0.226
1997,87,11,0,20,90.0,37.4,7.39,94,85.9,15.0,0.259
1998,40,16,1,17,28.9,18.3,3.62,21,87.5,15.0,0.194


In [7]:
X = d.drop(columns=['stars'])
y = d['stars']

knn = KNeighborsClassifier()
knn.fit(X, y)
print('')

In [12]:
def imputation(row):
    if row['stars'] != row['stars']:
        X_new = row.to_frame().T
        X_new = X_new.drop(columns = ['stars'])
        imput = knn.predict(X_new)[0]
        return imput
    else:
        return row['stars']

In [13]:
df['stars'] = df.apply(imputation, axis=1)
df

,season,weekday,guests,nights,stars,dist_center,breakfast,early_book,cancel_days,price
0,3,2,1,8,3,4.58,0,73,-1,3369.41
1,4,6,2,19,4,11.75,0,26,-1,3685.47
2,1,2,5,19,4,11.71,0,162,-1,2552.05
3,3,5,4,17,3,7.52,0,61,-1,3417.73
4,3,3,2,18,4,1.72,1,262,-1,4954.84
...,...,...,...,...,...,...,...,...,...,...
8995,4,3,1,1,4,6.10,0,258,-1,5244.75
8996,1,1,2,2,4,10.43,1,175,-1,2443.73
8997,2,2,4,11,3,6.02,1,145,-1,1771.46
8998,1,0,5,12,4,2.71,1,130,-1,3064.14


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9000 entries, 0 to 8999
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   season       9000 non-null   int64  
 1   weekday      9000 non-null   int64  
 2   guests       9000 non-null   int64  
 3   nights       9000 non-null   int64  
 4   stars        9000 non-null   object 
 5   dist_center  9000 non-null   float64
 6   breakfast    9000 non-null   int64  
 7   early_book   9000 non-null   int64  
 8   cancel_days  9000 non-null   int64  
 9   price        9000 non-null   float64
dtypes: float64(2), int64(7), object(1)
memory usage: 703.3+ KB


In [2]:
import numpy as np
import pandas as pd

def generate_fast_route(n=5000, seed=42):
    rng = np.random.default_rng(seed)

    df = pd.DataFrame({
        'distance': rng.uniform(50, 1500, n).round(1),
        'cargo_weight': rng.uniform(0.5, 20, n).round(1),
        'cargo_type': rng.choice([1, 2, 3, 4], n, p=[0.15, 0.20, 0.45, 0.20]),
        'departure_hour': rng.integers(0, 24, n),
        'day_of_week': rng.integers(0, 7, n),
        'season': rng.integers(1, 5, n),
        'weather_score': rng.integers(1, 11, n),
        'driver_exp': rng.integers(1, 26, n),
        'traffic_index': rng.uniform(1, 10, n).round(1)
    })

    # Таргет 1: Расход топлива
    fuel = (26.5 + (df['cargo_weight'] - 10) * 0.35
            + (df['traffic_index'] - 5) * 0.5
            + (10 - df['weather_score']) * 0.3
            + np.select([df['cargo_type'] == 1, df['cargo_type'] == 4], [1.2, 2.5], 0)
            + np.where(df['season'] == 1, 2.5, np.where(df['season'] == 3, -1.0, 0))
            - df['driver_exp'] * 0.04
            + rng.normal(0, 1.2, n))
    df['fuel_consumption'] = fuel.clip(14, 48).round(1)

    # Таргет 2: Время доставки
    time = (df['distance'] / 57
            + (df['traffic_index'] - 5) * 0.2
            + (10 - df['weather_score']) * 0.15
            + np.select([df['cargo_type'] == 1, df['cargo_type'] == 4], [0.5, 0.8], 0)
            + np.where((df['departure_hour'] < 5) | (df['departure_hour'] > 22), 0.4, 0)
            - df['driver_exp'] * 0.015
            + rng.normal(0, 0.9, n))
    df['delivery_time'] = time.clip(1, 40).round(2)

    # Таргет 3: Износ
    wear = (0.04 + df['cargo_weight'] * 0.003
            + df['distance'] * 0.000035
            + (10 - df['weather_score']) * 0.004
            + (df['traffic_index'] - 5) * 0.003
            + np.where(df['cargo_type'] == 4, 0.02, 0)
            + np.where(df['season'] == 1, 0.02, np.where(df['season'] == 3, -0.005, 0))
            - df['driver_exp'] * 0.0008
            + rng.normal(0, 0.012, n))
    df['wear_index'] = wear.clip(0, 1).round(3)

    return df

# Использование:
# df = generate_fast_route(10000)
# print(df.head())


In [3]:
df = generate_fast_route(10000)
df

,distance,cargo_weight,cargo_type,departure_hour,day_of_week,season,weather_score,driver_exp,traffic_index,fuel_consumption,delivery_time,wear_index
0,1172.2,14.6,3,2,5,2,2,19,1.9,26.3,19.72,0.153
1,686.4,14.4,4,8,0,2,7,23,2.2,29.6,12.40,0.103
2,1295.0,4.5,1,17,2,1,1,15,6.5,29.5,24.64,0.147
3,1061.2,1.2,2,10,1,1,8,11,7.4,28.5,19.88,0.088
4,186.6,6.4,3,4,3,1,1,1,7.8,32.2,5.14,0.131
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,697.8,16.3,3,10,5,3,10,19,2.5,26.3,12.47,0.084
9996,498.6,10.8,3,21,4,2,1,24,2.4,26.3,9.54,0.118
9997,289.8,17.0,3,6,2,1,6,22,7.8,33.7,4.18,0.133
9998,591.4,13.6,4,4,0,3,1,25,4.6,29.6,12.17,0.127


In [4]:
a = df.iloc[range(9000)]
a

,distance,cargo_weight,cargo_type,departure_hour,day_of_week,season,weather_score,driver_exp,traffic_index,fuel_consumption,delivery_time,wear_index
0,1172.2,14.6,3,2,5,2,2,19,1.9,26.3,19.72,0.153
1,686.4,14.4,4,8,0,2,7,23,2.2,29.6,12.40,0.103
2,1295.0,4.5,1,17,2,1,1,15,6.5,29.5,24.64,0.147
3,1061.2,1.2,2,10,1,1,8,11,7.4,28.5,19.88,0.088
4,186.6,6.4,3,4,3,1,1,1,7.8,32.2,5.14,0.131
...,...,...,...,...,...,...,...,...,...,...,...,...
8995,1403.4,17.8,3,0,0,4,10,7,8.9,32.6,24.93,0.151
8996,1262.4,9.8,1,14,2,3,4,21,1.0,26.8,22.17,0.106
8997,722.9,9.4,4,1,0,1,8,7,1.9,31.6,13.02,0.114
8998,1460.8,13.3,3,8,4,1,2,6,9.5,34.1,28.41,0.206


In [5]:
b = a[a.columns[:-2]]
b

,distance,cargo_weight,cargo_type,departure_hour,day_of_week,season,weather_score,driver_exp,traffic_index,fuel_consumption
0,1172.2,14.6,3,2,5,2,2,19,1.9,26.3
1,686.4,14.4,4,8,0,2,7,23,2.2,29.6
2,1295.0,4.5,1,17,2,1,1,15,6.5,29.5
3,1061.2,1.2,2,10,1,1,8,11,7.4,28.5
4,186.6,6.4,3,4,3,1,1,1,7.8,32.2
...,...,...,...,...,...,...,...,...,...,...
8995,1403.4,17.8,3,0,0,4,10,7,8.9,32.6
8996,1262.4,9.8,1,14,2,3,4,21,1.0,26.8
8997,722.9,9.4,4,1,0,1,8,7,1.9,31.6
8998,1460.8,13.3,3,8,4,1,2,6,9.5,34.1


In [6]:
# b.to_csv('data/logistics.csv', index = False)

In [7]:
c = a[a.columns[-2:]]
c

,delivery_time,wear_index
0,19.72,0.153
1,12.40,0.103
2,24.64,0.147
3,19.88,0.088
4,5.14,0.131
...,...,...
8995,24.93,0.151
8996,22.17,0.106
8997,13.02,0.114
8998,28.41,0.206


In [8]:
# c.to_csv('data/logistics_extra.csv', index = False)

In [9]:
d = df.iloc[range(9000, len(df))]
d

,distance,cargo_weight,cargo_type,departure_hour,day_of_week,season,weather_score,driver_exp,traffic_index,fuel_consumption,delivery_time,wear_index
9000,1316.8,0.7,3,6,3,3,6,8,9.6,25.5,23.86,0.096
9001,406.8,19.7,4,15,4,2,5,16,6.2,34.2,9.15,0.137
9002,759.0,18.9,2,18,6,4,3,6,6.0,31.2,13.26,0.133
9003,93.1,6.0,1,22,2,1,6,24,1.3,27.9,1.54,0.046
9004,62.9,11.4,4,23,3,2,5,6,4.3,32.1,1.00,0.122
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,697.8,16.3,3,10,5,3,10,19,2.5,26.3,12.47,0.084
9996,498.6,10.8,3,21,4,2,1,24,2.4,26.3,9.54,0.118
9997,289.8,17.0,3,6,2,1,6,22,7.8,33.7,4.18,0.133
9998,591.4,13.6,4,4,0,3,1,25,4.6,29.6,12.17,0.127


In [10]:
e = d[d.columns[:-3]]
e

,distance,cargo_weight,cargo_type,departure_hour,day_of_week,season,weather_score,driver_exp,traffic_index
9000,1316.8,0.7,3,6,3,3,6,8,9.6
9001,406.8,19.7,4,15,4,2,5,16,6.2
9002,759.0,18.9,2,18,6,4,3,6,6.0
9003,93.1,6.0,1,22,2,1,6,24,1.3
9004,62.9,11.4,4,23,3,2,5,6,4.3
...,...,...,...,...,...,...,...,...,...
9995,697.8,16.3,3,10,5,3,10,19,2.5
9996,498.6,10.8,3,21,4,2,1,24,2.4
9997,289.8,17.0,3,6,2,1,6,22,7.8
9998,591.4,13.6,4,4,0,3,1,25,4.6


In [11]:
# e.to_csv('data/logistics_new.csv', index = False)

In [13]:
f = introduce_missing(df=b, column='cargo_weight', missing_rate=0.05, random_state=42)
f.info()

✓ Добавлено 450 пропусков (5.0%) в столбец 'cargo_weight'.
<class 'pandas.core.frame.DataFrame'>
Int64Index: 9000 entries, 0 to 8999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   distance          9000 non-null   float64
 1   cargo_weight      8550 non-null   float64
 2   cargo_type        9000 non-null   int32  
 3   departure_hour    9000 non-null   int64  
 4   day_of_week       9000 non-null   int64  
 5   season            9000 non-null   int64  
 6   weather_score     9000 non-null   int64  
 7   driver_exp        9000 non-null   int64  
 8   traffic_index     9000 non-null   float64
 9   fuel_consumption  9000 non-null   float64
dtypes: float64(4), int32(1), int64(5)
memory usage: 996.3 KB


In [14]:
# f.to_csv('data/logistics_missing.csv', index=False)

In [15]:
g = introduce_missing(df=e, column='cargo_weight', missing_rate=0.05, random_state=42)
g.info()

✓ Добавлено 50 пропусков (5.0%) в столбец 'cargo_weight'.
<class 'pandas.core.frame.DataFrame'>
Int64Index: 1000 entries, 9000 to 9999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   distance        1000 non-null   float64
 1   cargo_weight    950 non-null    float64
 2   cargo_type      1000 non-null   int32  
 3   departure_hour  1000 non-null   int64  
 4   day_of_week     1000 non-null   int64  
 5   season          1000 non-null   int64  
 6   weather_score   1000 non-null   int64  
 7   driver_exp      1000 non-null   int64  
 8   traffic_index   1000 non-null   float64
dtypes: float64(3), int32(1), int64(5)
memory usage: 106.5 KB


In [16]:
# g.to_csv('data/logistics_new_missing.csv', index=False)

In [17]:
h = introduce_missing(df=f, column='distance', missing_rate=0.15, random_state=42)
h.info()

✓ Добавлено 1350 пропусков (15.0%) в столбец 'distance'.
<class 'pandas.core.frame.DataFrame'>
Int64Index: 9000 entries, 0 to 8999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   distance          7650 non-null   float64
 1   cargo_weight      8550 non-null   float64
 2   cargo_type        9000 non-null   int32  
 3   departure_hour    9000 non-null   int64  
 4   day_of_week       9000 non-null   int64  
 5   season            9000 non-null   int64  
 6   weather_score     9000 non-null   int64  
 7   driver_exp        9000 non-null   int64  
 8   traffic_index     9000 non-null   float64
 9   fuel_consumption  9000 non-null   float64
dtypes: float64(4), int32(1), int64(5)
memory usage: 996.3 KB


In [18]:
# h.to_csv('data/logistics_missing_2.csv', index=False)

In [19]:
w = introduce_missing(df=g, column='distance', missing_rate=0.15, random_state=42)
w.info()

✓ Добавлено 150 пропусков (15.0%) в столбец 'distance'.
<class 'pandas.core.frame.DataFrame'>
Int64Index: 1000 entries, 9000 to 9999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   distance        850 non-null    float64
 1   cargo_weight    950 non-null    float64
 2   cargo_type      1000 non-null   int32  
 3   departure_hour  1000 non-null   int64  
 4   day_of_week     1000 non-null   int64  
 5   season          1000 non-null   int64  
 6   weather_score   1000 non-null   int64  
 7   driver_exp      1000 non-null   int64  
 8   traffic_index   1000 non-null   float64
dtypes: float64(3), int32(1), int64(5)
memory usage: 106.5 KB


In [20]:
# w.to_csv('data/logistics_new_missing_2.csv', index=False)

In [2]:
import numpy as np
import pandas as pd

np.random.seed(42)  # для воспроизводимости

n_samples = 10000

# --- Предикторы ---

# Тип абонемента: 1–5, с перекосом в долгосрочные (люди чаще берут на 6/12 мес)
contract_type = np.random.choice([1, 2, 3, 4, 5], size=n_samples, p=[0.05, 0.1, 0.2, 0.3, 0.35])

# Дней до окончания абонемента: равномерное 0–60
days_to_expiry = np.random.randint(0, 61, size=n_samples)

# Закреплён ли персональный тренер: зависит от типа контракта (дольше — чаще тренер)
trainer_prob = 0.1 + 0.08 * (contract_type - 1)  # от ~0.1 до ~0.42
trainer_assigned = (np.random.rand(n_samples) < trainer_prob).astype(int)

# Дней с последнего визита: экспоненциальное распределение, обрезаем до 90
# У активных клиентов меньше дней, у «забросивших» — больше
base_days = np.random.exponential(scale=15, size=n_samples)
days_since_last_visit = np.clip(base_days, 0, 90).astype(int)

# Посещения за последние 30 дней: Пуассон, зависит от «свежести» последнего визита и тренера
# Чем свежее визит и есть тренер — тем больше посещений
visits_rate = 8.0 / (1 + days_since_last_visit / 15) * (1 + 0.4 * trainer_assigned)
visits_last_30d = np.random.poisson(lam=visits_rate, size=n_samples)
visits_last_30d = np.clip(visits_last_30d, 0, 20)

# Средняя длительность тренировки: нормальное распределение, зависит от активности
avg_session_len_min = np.random.normal(loc=60, scale=15, size=n_samples)
avg_session_len_min = np.clip(avg_session_len_min, 30, 120)

# Групповые занятия за 60 дней: Пуассон, зависит от посещений и тренера
classes_rate = 0.6 * visits_last_30d * (1 + 0.3 * trainer_assigned)
classes_attended = np.random.poisson(lam=classes_rate, size=n_samples)
classes_attended = np.clip(classes_attended, 0, 15)

# Пропущенные забронированные групповые занятия за 30 дней: Пуассон
# Больше пропусков у тех, кто реже ходит и у кого нет тренера
missed_rate = 2.0 / (1 + visits_last_30d / 5) * (1 + 0.5 * (1 - trainer_assigned))
missed_classes = np.random.poisson(lam=missed_rate, size=n_samples)
missed_classes = np.clip(missed_classes, 0, 10)

# Промокоды за 90 дней: биномиальное (шанс использовать), зависит от типа контракта
promo_p = 0.15 + 0.05 * (contract_type == 5).astype(float)
promo_used_last_90d = np.random.binomial(n=5, p=promo_p, size=n_samples)

# Оценка удовлетворённости: нормальное, зависит от вовлечённости и пропусков
feedback_base = 4.0 + 0.03 * visits_last_30d - 0.1 * missed_classes
feedback_score = np.random.normal(loc=feedback_base, scale=0.6, size=n_samples)
feedback_score = np.clip(feedback_score, 1, 5)

# --- Таргеты (логика формирования) ---

# Базовая вероятность оттока: растёт, если скоро истекает контракт и давно не был
churn_logit = -2.5 + 0.04 * days_to_expiry + 0.03 * days_since_last_visit - 0.15 * visits_last_30d
churn_prob = 1 / (1 + np.exp(-churn_logit))
churn_30d = (np.random.rand(n_samples) < churn_prob).astype(int)

# Низкая вовлечённость: мало посещений, зависит от тренера и длительности сессии
low_eng_logit = 1.2 - 0.25 * visits_last_30d + 0.1 * (avg_session_len_min < 45) - 0.3 * trainer_assigned
low_eng_prob = 1 / (1 + np.exp(-low_eng_logit))
low_engagement_30d = (np.random.rand(n_samples) < low_eng_prob).astype(int)
# Принудительно: если посещений < 2 — точно низкая вовлечённость
low_engagement_30d[visits_last_30d < 2] = 1

# Апгрейд/продление: выше у лояльных, с тренером, с высокой оценкой и без оттока
upgrade_logit = -1.5 + 0.15 * feedback_score + 0.2 * trainer_assigned + 0.1 * visits_last_30d - 1.5 * churn_30d
upgrade_prob = 1 / (1 + np.exp(-upgrade_logit))
upgrade_30d = (np.random.rand(n_samples) < upgrade_prob).astype(int)

# Сборка DataFrame
df = pd.DataFrame({
    'days_since_last_visit': days_since_last_visit,
    'visits_last_30d': visits_last_30d,
    'avg_session_len_min': avg_session_len_min,
    'classes_attended': classes_attended,
    'trainer_assigned': trainer_assigned,
    'promo_used_last_90d': promo_used_last_90d,
    'contract_type': contract_type,
    'days_to_expiry': days_to_expiry,
    'feedback_score': feedback_score,
    'missed_classes': missed_classes,
    'churn_30d': churn_30d,
    'low_engagement_30d': low_engagement_30d,
    'upgrade_30d': upgrade_30d
})

# Проверка распределения таргетов
print("Распределение таргетов:")
print(df[['churn_30d', 'low_engagement_30d', 'upgrade_30d']].mean())

# Сохранение в CSV (опционально)
# df.to_csv('fitness_churn_dataset.csv', index=False)

df.head()


Распределение таргетов:
churn_30d             0.1942
low_engagement_30d    0.4692
upgrade_30d           0.3752
dtype: float64


,days_since_last_visit,visits_last_30d,avg_session_len_min,classes_attended,trainer_assigned,promo_used_last_90d,contract_type,days_to_expiry,feedback_score,missed_classes,churn_30d,low_engagement_30d,upgrade_30d
0,39,2,65.300359,0,0,1,4,54,4.810134,0,0,0,1
1,16,2,64.172864,2,0,1,5,31,4.633475,2,0,0,1
2,24,4,67.337259,1,1,3,5,35,5.000000,1,0,0,0
3,0,8,43.501275,1,0,0,4,49,5.000000,1,0,0,1
4,13,4,66.596392,3,0,1,3,36,4.746998,0,1,0,1


In [6]:
a = df.iloc[range(9000)]
a

b = a[a.columns[:-2]]
b

# b.to_csv('data/fitnes.csv', index = False)

c = a[a.columns[-2:]]
c

# c.to_csv('data/fitnes_extra.csv', index = False)

d = df.iloc[range(9000, len(df))]
d

e = d[d.columns[:-3]]
e

# e.to_csv('data/fitnes_new.csv', index = False)

f = introduce_missing(df=b, column='avg_session_len_min', missing_rate=0.05, random_state=42)
f.info()

# f.to_csv('data/fitnes_missing.csv', index=False)

g = introduce_missing(df=e, column='avg_session_len_min', missing_rate=0.05, random_state=42)
g.info()

# g.to_csv('data/fitnes_new_missing.csv', index=False)

h = introduce_missing(df=f, column='feedback_score', missing_rate=0.15, random_state=42)
h.info()

# h.to_csv('data/fitnes_missing_2.csv', index=False)

w = introduce_missing(df=g, column='feedback_score', missing_rate=0.15, random_state=42)
w.info()

# w.to_csv('data/fitnes_new_missing_2.csv', index=False)

✓ Добавлено 450 пропусков (5.0%) в столбец 'avg_session_len_min'.
<class 'pandas.core.frame.DataFrame'>
Int64Index: 9000 entries, 0 to 8999
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   days_since_last_visit  9000 non-null   int32  
 1   visits_last_30d        9000 non-null   int32  
 2   avg_session_len_min    8550 non-null   float64
 3   classes_attended       9000 non-null   int32  
 4   trainer_assigned       9000 non-null   int32  
 5   promo_used_last_90d    9000 non-null   int32  
 6   contract_type          9000 non-null   int32  
 7   days_to_expiry         9000 non-null   int32  
 8   feedback_score         9000 non-null   float64
 9   missed_classes         9000 non-null   int32  
 10  churn_30d              9000 non-null   int32  
dtypes: float64(2), int32(9)
memory usage: 785.4 KB
✓ Добавлено 50 пропусков (5.0%) в столбец 'avg_session_len_min'.
<class 'pandas.core.frame.DataFram